In [1]:
import pandas as pd
import numpy as np

excel_file = "SC31_annotations_binary-table_cutted.xlsx"

# load the Excel
df = pd.read_excel(excel_file)
print(df.columns)
print("Absence unique values:", df["Absence"].unique())

Index(['No. Frame', 'Time (25 fps) ms', 'Time (30 fps) s', 'Absence',
       'Epaules', 'Main - Bras', 'Tête', 'Yeux', 'Visage', 'Bassin - Tronc',
       'Jambes - Pieds', 'Phonique', 'Vocaux', 'Unnamed: 13', 'Cam',
       'Tic_complexe', 'Unnamed: 16', 'Commentaires', 'Unnamed: 18',
       'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21'],
      dtype='object')
Absence unique values: [1 0]


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("SC31_annotations_binary-table_cutted.xlsx")

# bodyparts columns
movement_cols = ["Epaules", "Main - Bras", "Tête", "Yeux", "Visage",
                 "Bassin - Tronc", "Jambes - Pieds", "Phonique", "Vocaux"]

# time we think there is a tic
target_time = 348.876

# find the index of the target line
idx = (df["Time (30 fps) s"] - target_time).abs().idxmin()

print("Index du début attendu :", idx)
print(df.loc[idx-35:idx+5, ["Time (30 fps) s", "Absence"] + movement_cols])

Index du début attendu : 10572
       Time (30 fps) s  Absence  Epaules  Main - Bras  Tête  Yeux  Visage  \
10537          347.721        1        0            0     0     0       0   
10538          347.754        1        0            0     0     0       0   
10539          347.787        1        0            0     0     0       0   
10540          347.820        1        0            0     0     0       0   
10541          347.853        1        0            0     0     0       0   
10542          347.886        1        0            0     0     0       0   
10543          347.919        1        0            0     0     0       0   
10544          347.952        1        0            0     0     0       0   
10545          347.985        1        0            0     0     0       0   
10546          348.018        1        0            0     0     0       0   
10547          348.051        1        0            0     0     0       0   
10548          348.084        1        0     

In [12]:
import pandas as pd
import numpy as np


def extract_tics_from_excel(excel_file, phase_start_s, phase_end_s, min_absence_frames=30):

    """
    ----------
    Purpose
    ----------
    Extract tic intervals from a specified time window (phase).

    ----------
    Parameters
    ----------
    excel_file : str
        Path to the Excel annotation file.
    phase_start_s : float
        Start time of the phase to analyze (seconds).
    phase_end_s : float
        End time of the phase to analyze (seconds).
    min_absence_frames : int
        Minimum number of consecutive "Absence = 1" frames before AND after a tic.

    ----------
    Returns
    ----------
    tics : list of tuples
        [(start_time_s, end_time_s), ...] only inside the selected phase.
    """

    # load the Excel
    df = pd.read_excel(excel_file)
    # print(df.columns)


    # check the required columns
    time_col = "Time (30 fps) s"
    if time_col not in df.columns:
        raise ValueError(f"Missing required column: {time_col}")
    if "Absence" not in df.columns:
        raise ValueError("Missing required column 'Absence'.")
    times = df[time_col].values
    absence = df["Absence"].values

    # define explicitly the columns corresponding to the tics bodyparts
    movement_cols = ["Epaules", "Main - Bras", "Tête", "Yeux", "Visage", "Bassin - Tronc", "Jambes - Pieds", "Phonique", "Vocaux"]
    # convert into a numpy array
    movement_data = df[movement_cols].values
    # count the number of active bodyparts columns per frame
    movement_sum = movement_data.sum(axis=1)

    # tic frames : Absence = 0 & movement activity = 1+
    # is_tic_frame = (absence == 0) & (movement_sum > 0)

    tics = []
    n = len(df)
    i = 0


    # main detection loop
    # iterate through all frames until the end
    while i < n:

        # -------------------- TIC CONDITION 1 : 30 frames of "Absence" before --------------------
        if i >= min_absence_frames:
            before_ok = np.all((absence[i-min_absence_frames:i] == 1) & (movement_sum[i-min_absence_frames:i] == 0))
        else:
            before_ok = False

            i += 1 # otherwise, move to the next frame
            continue # and skip the rest of this iteration

        # -------------------- TIC CONDITION 2 : frame with "Absence"=0 and BodyParts=1 --------------------
        if before_ok and absence[i] == 0 and movement_sum[i] > 0:
            start_idx = i
            start_time = times[start_idx]

            # search the end while in the condition 2
            j = i
            while j < n and (absence[j] == 0 and movement_sum[j] > 0):
                j += 1 # move forward frame by frame

            end_idx = j - 1
            end_time_candidate = times[end_idx]

            # -------------------- TIC CONDITION 3 : 30 frames of "Absence" after --------------------
            # if j + min_absence_frames <= n:
            #     after_ok = np.all((absence[j:j+min_absence_frames] == 1) & (movement_sum[j:j+min_absence_frames] == 0))
            # else:
            #    after_ok = False

            while j < n and not ((absence[j] == 1) and (movement_sum[j] == 0)):
                j += 1
            
            # measure how many consecutive frames are pure absence
            after_len = 0
            k = j
            while k < n and (absence[k] == 1) and (movement_sum[k] == 0):
                k += 1
            after_len = k - j

            # verify if the lenght is enough
            after_ok = after_len >= min_absence_frames

            
            if after_ok:
                # add & save the tic interval as a (start, end) pair
                tics.append((start_time, end_time_candidate))
                # i = end_idx + min_absence_frames
                i = k
                continue # and restart detection from there

        # if no tic in this frame, just move to next frame
        i += 1
    
    # filter tics inside phase
    tics_in_phase = []
    for start, end in tics:
        # keep the tic if it intersects the phase
        if end >= phase_start_s and start <= phase_end_s:
            tics_in_phase.append((start, end))

    # return all detected tic intervals
    return tics_in_phase



# small test if run directly (recommended!)
if __name__ == "__main__":
    # Filename of your real Excel annotations file
    test_file = "SC31_annotations_binary-table_cutted.xlsx"

    phase_start = 345.972
    phase_end = 970.167
    
    try:
        tics = extract_tics_from_excel(excel_file=test_file, phase_start_s=phase_start, phase_end_s=phase_end, min_absence_frames=30)
        print("\nTics detected inside selected phase:")
        if not tics:
            print(" No tics detected in this phase.")
        else:
            for start, end in tics:
                print(f" - Tic from {start:.3f}s to {end:.3f}s")
    except Exception as e:
        print(f"Error: {e}")


Tics detected inside selected phase:
 - Tic from 426.690s to 437.217s
 - Tic from 454.245s to 455.697s
 - Tic from 457.875s to 458.271s
 - Tic from 459.690s to 459.987s
 - Tic from 773.124s to 774.576s
 - Tic from 830.643s to 844.602s
 - Tic from 848.397s to 896.577s


In [ ]:
import pandas as pd
import numpy as np

def extract_tics_from_excel(excel_file, phase_start_s, phase_end_s, min_absence_frames=30):
    """
    Robust tic extraction using run-length style checks.
    Returns tic intervals that intersect the requested phase.
    """
    df = pd.read_excel(excel_file)

    time_col = "Time (30 fps) s"
    if time_col not in df.columns:
        raise ValueError(f"Missing required column: {time_col}")
    if "Absence" not in df.columns:
        raise ValueError("Missing required column 'Absence'.")

    times = df[time_col].values
    absence = df["Absence"].values.astype(int)

    movement_cols = ["Epaules", "Main - Bras", "Tête", "Yeux", "Visage",
                     "Bassin - Tronc", "Jambes - Pieds", "Phonique", "Vocaux"]
    # ensure columns exist
    missing = [c for c in movement_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing movement columns in excel: {missing}")

    movement_sum = df[movement_cols].values.sum(axis=1).astype(int)

    # boolean arrays
    absence_block = (absence == 1) & (movement_sum == 0)   # True when frame belongs to "pure absence" block
    is_movement_frame = (absence == 0) & (movement_sum > 0)  # True when frame shows movement

    n = len(df)

    # --- helper: run-length encoding for boolean array ---
    # returns lists: values, start_indices, lengths
    def rle_bool(arr):
        if len(arr) == 0:
            return np.array([], dtype=bool), np.array([], dtype=int), np.array([], dtype=int)
        a = np.asarray(arr, dtype=int)
        diff = np.diff(a)
        run_starts = np.flatnonzero(diff != 0) + 1
        run_starts = np.r_[0, run_starts]
        run_ends = np.r_[run_starts[1:], len(a)]
        lengths = run_ends - run_starts
        values = a[run_starts].astype(bool)
        return values, run_starts, lengths

    vals, starts, lengths = rle_bool(absence_block)
    # build array that maps each index to (run_value, run_start, run_len)
    # we only need runs where values == True
    # create an array prev_absence_len_at_idx: length of consecutive True block that ENDS at idx (if any)
    prev_absence_len_end_at = np.zeros(n, dtype=int)  # length of absence_block run ending at this idx (0 if not end)
    run_id = 0
    for val, s, L in zip(vals, starts, lengths):
        if val:  # this run is a True run in absence_block
            end = s + L - 1
            # mark all indices in the run with the run length (useful later)
            prev_absence_len_end_at[s:end+1] = L

    # For convenience, also compute for each index the length of the continuous absence_block immediately BEFORE index i
    # i.e. number of consecutive absence_block True frames that end at i-1
    prev_absence_before_idx = np.zeros(n, dtype=int)
    # we can compute by shifting prev_absence_len_end_at
    prev_absence_before_idx[1:] = prev_absence_len_end_at[:-1]

    # similarly compute next_absence_len_starting_at: length of absence_block run starting at index i (0 if not start)
    next_absence_len_starting_at = np.zeros(n, dtype=int)
    for val, s, L in zip(vals, starts, lengths):
        if val:
            next_absence_len_starting_at[s] = L
            # also fill all positions in run if needed:
            next_absence_len_starting_at[s:s+L] = L

    # Now detection:
    tics = []
    i = 0
    while i < n:
        if not is_movement_frame[i]:
            i += 1
            continue

        # candidate start: ensure previous run length (ending at i-1) >= min_absence_frames
        prev_len = prev_absence_before_idx[i]  # number of consecutive absence_block frames immediately before i
        if prev_len < min_absence_frames:
            i += 1
            continue

        # find the last movement frame of this tic (extend j while movement frames)
        j = i
        while j < n and is_movement_frame[j]:
            j += 1
        end_idx = j - 1

        # check after: need at least min_absence_frames starting at j
        after_len = 0
        if j < n:
            # number of consecutive absence_block starting at j
            # after_len = next_absence_len_starting_at[j]
            after_len = next_absence_len_starting_at[j] if j < n else 0
        if after_len < min_absence_frames:
            # not enough absence after -> skip this candidate, continue after the movement block
            i = j
            continue

        # valid tic -> record start time (time at first movement frame i)
        start_time = times[i]
        end_time = times[end_idx]
        tics.append((start_time, end_time))

        # advance i safely past the validated absence block to avoid re-detecting overlapping
        i = j + after_len  # jump after the confirmed absence block

    # filter to tics that intersect with requested phase
    tics_in_phase = [(s, e) for s, e in tics if (e >= phase_start_s and s <= phase_end_s)]
    return tics_in_phase



# small test if run directly (recommended!)
if __name__ == "__main__":
    # Filename of your real Excel annotations file
    test_file = "SC31_annotations_binary-table_cutted.xlsx"

    phase_start = 345.972
    phase_end = 970.167
    
    try:
        tics = extract_tics_from_excel(excel_file=test_file, phase_start_s=phase_start, phase_end_s=phase_end, min_absence_frames=30)
        print("\nTics detected inside selected phase:")
        if not tics:
            print(" No tics detected in this phase.")
        else:
            for start, end in tics:
                print(f" - Tic from {start:.3f}s to {end:.3f}s")
    except Exception as e:
        print(f"Error: {e}")



Tics detected inside selected phase:
 - Tic from 426.690s to 437.217s
 - Tic from 457.875s to 458.271s
 - Tic from 459.690s to 459.987s
 - Tic from 773.124s to 774.576s
 - Tic from 830.643s to 844.602s
 - Tic from 848.397s to 896.577s
